## Reinforcement Learning

- RL is the study of agents and how they learn by trial and error.
  
- RL is a class of methods for solving various kinds of sequential decision making tasks

  
- It formalizes the idea that rewarding or punishing an agent for its behavior makes it more likely to repeat or forego that behavior in the future.

<center><img src="img/fig98.png" width=400 height=400 /></center>


## 1. Applications

- Equilibre Technologies: RL for algorithmic trading
- Netflix: "GenPage" for End-to-End Homepage Generation
- Robotics
- Many other real-world applications

## 2. States and Observations

- A state $s$ is a complete description of the state of the world.
    
        
    - **Simple Grid:** The state could simply be the agent's coordinates, like $(x,y)$. The state space is finite and discrete.
    
    - **Robotics:** the state of a robot might be represented by its joint angles, velocities, sensor readings, and target location. This often involves continuous values.

- An observation $o$ is a partial description of a state, which may omit information.

- represent states and observations by a real-valued vector, matrix, or higher-order tensor.


## 3. Action Spaces

- Different environments allow different kinds of actions.
- The set of all valid actions in a given environment.
    - Some environments, like Atari, have **discrete action spaces**.
    - Other environments, like where the agent controls a robot in a physical world, have **continuous action spaces**.

## 4. Trajectories

A trajectory $\tau$ is a sequence of states and actions in the world,

$$\tau = (s_0, a_0, s_1, a_1, ...).$$

## 5. Reward

It depends on **the current state** of the world, **the action** just taken, and **the next state** of the world:

$$r_t = R(s_t, a_t, s_{t+1})$$


- **Rewards** are scalar values (single numbers). They can be positive, negative, or zero.

    - Robot Control: Positive reward for reaching the target, negative reward for hitting obstacles or
      consuming too much energy.
      
<break>
    
- **The goal of the agent** is to **maximize** some notion of cumulative reward over a trajectory

  - finite-horizon undiscounted return
$$R(\tau) = \sum_{t=0}^T r_t.$$

  - infinite-horizon discounted return  $ \gamma \in (0,1)$:
$$ R(\tau) = \sum_{t=0}^{\infty} \gamma^t r_t.$$
       - is the sum of all rewards ever obtained by the agent, but discounted by how far off in the future they’re obtained

## 6. Policies: Mapping States to Actions


A policy is a rule used by an agent to decide what actions to take.

- It can be **deterministic**, directly specifies the action the agent will take for each state:

$$a_t = \mu_{\theta}(s_t) $$

- or it may be **stochastic**, defines a probability distribution over actions for each state:
    - it tells us the probability of taking each possible action $a$ when in state $s$ 

$$a_t \sim \pi_{\theta}(\cdot | s_t).$$

<center><img src="img/fig01.png" width=300 height=300 /></center>

- In deep RL: we deal with parameterized policies $\theta$
- Policies whose outputs are computable functions that depend on a set of parameters (eg the weights and biases of a neural network)

#### build a simple deterministic policy for a continuous action space in PyTorch

In [ ]:
pi_net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, act_dim)
            )

### The Goal: Finding the Optimal Policy
The central objective in Reinforcement Learning is typically to find an optimal policy, denoted as $π^*$. 

An optimal policy is one that maximizes the expected cumulative reward the agent receives over the long run, starting from any state.

**The central optimization problem in RL can be expressed by**

$$
\pi^* = \arg \max_{\pi} \mathbb{E}_{\tau\sim\pi} [R(\tau)]
$$

>> Directly searching over all policies to maximize this expectation is intractable, because the space of trajectories is enormous (often infinite).

## 7. Types of RL Tasks: Episodic vs Continuing

**Continual task:** the agent can potentially interact with the environment forever.

    - typically use the discounted sum over an infinite horizon. 

**Episodic task:** the agent's interaction terminates once the system enters a terminal state or absorbing state
    
    - usually use the undiscounted sum of rewards per episode
    
    - may start a new episode from a new initial world state

## 8. Modeling Sequential Decision Making

- Markov Decision Processes (MDPs) provide the standard mathematical framework used throughout Reinforcement Learning.
- MDP is defined by a tuple containing five components: $(S,A,P,R,γ)$
    - State Space, Action space, Transition probability function, Reward function, and Discount factor

### 8.1 Value Functions

- As we formulated the RL problem using MDP. Now, how does an agent actually decide what to do in this environment? And how do we measure how good those decisions are in the long run?

- value functions quantify how good it is for an agent to be in a particular state $V^{π}(s)$
- or to take a specific action in a state $Q^{π}(s,a)$ while following a policy 
$π$

#### The Bellman equations for the optimal state-value are

$
V^*(s) = \max_a \mathbb{E}\big[r(s,a) + \gamma V^*(s')\big]
$

> the optimal value of a state $s$ equals the expected return obtained by taking the best possible action $a$ in that state, and then continuing optimally from the resulting state $s'$




#### The Bellman equations for the optimal action-value functions


$
Q^*(s,a) = \mathbb{E}\big[r(s,a) + \gamma \max_{a'} Q^*(s',a')\big]
$

> The optimal value of taking action $a$ in state $s$ is the expected immediate reward $r$ plus the discounted expected value of the best possible action $a'$ taken in the next state $s'$

## 9. Solving Bellman Equations

- **Dynamic Programming (DP):** depends on a complete and accurate model of the environment. Unrealistic


- **Monte Carlo Methods:** updates value estimates only after an entire episode completes.


- **Temporal-Difference Learning:** works well for problems with a manageable number of states

    - Monte Carlo and Temporal Difference learning rely on maintaining a table mapping each state or state-action pair to a value.
      
- **Value Function Approximation** using Neural Network


- **Deep Q-Networks**


- **Policy Gradient Methods**


<center><img src="img/fig99.png" width=550 height=550 /></center>


<center>A non-exhaustive, but useful taxonomy of algorithms in modern RL</center>

## 10. Temporal-Difference Learning (TD Learning)
**Learning from Incomplete Episodes**

Imagine being at state $S_{t}$, taking action $A_{t}$, receiving reward $R_{t+1}$, and landing in state $S_{t+1}$.

- **MC methods** would wait until the end of the episode time $T$ to compute the actual return $V(S_t) = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{T-t-1} R_T$. 

- **TD methods** form a target using only the next step's information: the immediate reward $R_{t+1}$ plus the discounted current (or estimate) of the next state's value, $\gamma V(S_{t+1})$.

    - The target for the update at time $t$ is $R_{t+1} + \gamma V(S_{t+1})$.

    - This process of updating an estimate based partially on another learned estimate is called **bootstrapping**


### SARSA: On-Policy TD Control

**SARSA** performs an update after each time step. The update rule for the action-value function $Q(S_t, A_t)$ is:
$$Q(S_{t},A_{t})\leftarrow Q(S_{t},A_{t})+\alpha \left[R_{t+1}+\gamma Q(S_{t+1},A_{t+1})-Q(S_{t},A_{t})\right]$$

- $Q(S_t, A_t)$ is the current estimate of the value of taking action $A_{t}$ in state $S_{t}$.
  
- $\alpha$ is the learning rate (e.g., $0.1$), controlling how much we update the estimate based on the new information.

  
- $R_{t+1}$ is the immediate reward received after taking action $A_{t}$.

  
- $\gamma$ is the discount factor (between $0$ and $1$), balancing immediate versus future rewards.

  
- $Q(S_{t+1}, A_{t+1})$ is the estimated value of the next state-action pair.

    - This is the bootstrapping part.
    - Unlike Monte Carlo, we use the current estimate of the value of the next action the agent will take according to its policy.



- $R_{t+1} + \gamma Q(S_{t+1}, A_{t+1})$ is the TD target. It represents a better estimate of $Q(S_t, A_t)$ based on the one-step experience.



- $R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)$ is the TD error, measuring the difference between the estimated value and the TD target.


#### Why is SARSA On-Policy?

- SARSA learns the value function $Q_π$ for the policy $π$ that the agent is currently following.

- The policy being followed includes the exploration steps (e.g., the random actions taken by $ϵ$-greedy).

### SARSA Algorithm

```text
Parameters: learning rate α > 0, discount factor γ ∈ [0, 1], exploration rate ε > 0

Initialize Q(s, a) for all s ∈ S, a ∈ A(s)
Ensure Q(terminal_state, ·) = 0

FOR each episode DO
    Initialize S                                  ▷ starting state
    Choose A from S using policy derived from Q   (e.g., ε-greedy)

    REPEAT                                        ▷ for each step of the episode
        Take action A, observe R, S'
        Choose A' from S' using policy derived from Q   (e.g., ε-greedy)

        ▷ SARSA Update
        Q(S, A) ← Q(S, A) + α · [ R + γ · Q(S', A') − Q(S, A) ]

        S ← S'                                    ▷ move to next state
        A ← A'                                    ▷ move to next action
    UNTIL S is terminal
END FOR
```

### Q-Learning: Off-Policy TD Control

- **"off-policy"** means the policy being learned about (**the target policy**) is different from the policy used to generate the behavior (**the behavior policy**).
  
- Q-Learning aims directly for the optimal action-value function $Q^*(S,A)$


- **Update Rule:** After taking action $A_t$ in state $S_t$ and observing the reward $R_{t+1}$ and the next state $S_{t+1}$,

    - Q-Learning updates its estimate $Q(S_t,A_t)$ using the following rule:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t)]$$

* $\max_{a} Q(S_{t+1}, a)$: This is the defining part of Q-Learning.
* Instead of using the Q-value of the actual next action taken ($A_{t+1}$, like in SARSA),
* Q-Learning uses the maximum possible Q-value for any action $a$ available in the next state $S_{t+1}$.


### Algorithm: Q-Learning

```text
Initialize Q(s, a) arbitrarily for all s in S, a in A(s)
(e.g., Q(s, a) = 0)
Initialize Q(terminal_state, .) = 0

Loop for each episode:
    Initialize S (starting state for the episode)
    Loop for each step of the episode:
        Choose A from S using policy derived from Q (e.g., epsilon-greedy)
        Take action A, observe R, S'

        # Core Q-Learning Update
        Q(S, A) <- Q(S, A) + alpha * [R + gamma * max_a Q(S', a) - Q(S, A)]

        S <- S'  # Move to the next state

        If S is terminal, break inner loop (end episode)
End Loop (episodes)
```

### On-Policy vs. Off-Policy Learning

#### SARSA (On-Policy):
* Learns the value function $Q^π$ for the policy $π$ the agent is currently following.
* If the agent explores using an $ϵ$-greedy strategy, SARSA learns the value of acting $ϵ$-greedily.
* The behavior policy and the target policy are the same.

#### Q-Learning (Off-Policy):

* Learns the value function $Q^*$ for the optimal greedy policy, regardless of the policy the agent is currently following to explore the environment.

* The behavior policy (e.g., $ϵ$-greedy) can be different from the target policy (which is always the greedy policy implied by $\max_a Q(S_{t+1},a)$.

### Behavioral Consequences

**"Cliff Walking" problem**

<center><img src="img/cliffwalk.png" width=500 height=500 /></center>


- Q-Learning learns the shortest path, right along the cliff edge. It assumes it will always act optimally, so it ignores the risk that a random move sends it over.

- SARSA learns a safer path further from the edge. Its updates include the occasional random exploratory move, so it "knows" that walking near the cliff sometimes ends badly.


If ε is gradually reduced toward 0, SARSA's policy becomes greedy and it also converges to the optimal policy.

## 11. Exploration vs. Exploitation

This tradeoff between exploring new strategies vs sticking to already known ones is called **the exploration-exploitation problem**

This is a key ingredient in RL problems and something that distinguishes RL problems from supervised machine learning.

- **Exploration** is exploring the environment by trying random actions in order to find more information about the environment.

- **Exploitation** is exploiting known information to maximize the reward.


Technically speaking, we want the agent **to find the global optimum, not a local one**.

In [ ]:
if random.uniform(0, 1) < epsilon:
    # Explore action space
    action = env.action_space.sample()
else:
    # Exploit learned values (or policy)
    action = agent.get_best_action(state)